# Lesson 2: Handoffs And Reservations

## Objective

Practice the Geond operating loop for safe parallel edits: review context, reserve a symbol, check conflicts, and leave a handoff.

## Prerequisites

- Complete Lesson 1 or run `uv run geond seed-sample` first.
- Local PostgreSQL should be reachable.
- Use the sample workspace unless you intentionally choose a different test workspace.

## Safety

This lesson reserves sample work only. Do not use private production workspace ids when experimenting.


In [ ]:
import json
import subprocess
from pathlib import Path

REPO = Path.cwd()
WORKSPACE_URI = "file:///sample/geond"


def run(args, check=True):
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=REPO, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result


def seed_workspace_id():
    result = run(["uv", "run", "geond", "seed-sample"])
    return json.loads(result.stdout)["workspace_id"]

## Run: create or refresh sample evidence

Expected outcome: Geond returns a workspace id you can use in reservation commands.


In [ ]:
workspace_id = seed_workspace_id()
workspace_id

## Run: review context before editing

Expected outcome: context review shows active coordination state for the requested symbol and intent.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "review-context",
        WORKSPACE_URI,
        "--intent",
        "Prepare a pair-coding change to build_answer",
        "--symbol",
        "build_answer",
        "--format",
        "markdown",
    ]
)

## Run: reserve and inspect conflict state

Expected outcome: the symbol reservation becomes visible, so another agent can detect overlap before editing.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "reserve-symbols",
        workspace_id,
        "--agent-name",
        "agent-a",
        "--symbol",
        "build_answer",
        "--purpose",
        "Lesson 2 sample reservation",
        "--ttl-minutes",
        "30",
    ]
)
run(["uv", "run", "geond", "conflicts", workspace_id, "--symbol", "build_answer"])

## Run: leave a structured handoff

Expected outcome: the next agent sees summary, next action, tested command, and remaining risk fields.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "record-handoff",
        workspace_id,
        "--from-agent",
        "agent-a",
        "--to-agent",
        "agent-b",
        "--summary",
        "Agent A reserved build_answer for a tutorial pair-coding check.",
        "--next-action",
        "Agent B should read conflicts before editing.",
        "--tested-command",
        "uv run geond conflicts <workspace> --symbol build_answer",
        "--risk",
        "This is sample tutorial state and should be cleaned up after use.",
    ]
)
run(["uv", "run", "geond", "list-handoffs", "--workspace-id-or-uri", workspace_id])

## Cleanup

Release or purge sample state when you are done:

```bash
uv run geond cleanup-reservations --workspace-id <workspace-id>
uv run geond purge-workspace file:///sample/geond --yes
```
